In [1]:
import os
import re
import json
from pathlib import Path
from collections import defaultdict

import numpy as np

import torch
import torch.nn as nn

import cv2
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from sklearn.model_selection import StratifiedGroupKFold

import torchvision.models.video as video_models
from torchvision.models.video import R2Plus1D_18_Weights

In [2]:
datasetpath = "/kaggle/input/datasets/sarbagyashakya/basketball-51-dataset/Basketball_51 dataset"
frames = 16
samplesperclass = 9999 # set to 9999 or some big number for the full dataset

workers = 4
batchsize = 32

savedir = "/kaggle/working/"
os.makedirs(savedir, exist_ok=True)
bestpath = os.path.join(savedir, "gamestrat.pt")

epochs = 10
patience = 2

# determinism stuff
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True

In [3]:
# normalize w/ the Kinetics-400 statistics used by the pretrained weights.
traintrans = v2.Compose([
    v2.Resize([112, 112], antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.43216, 0.394666, 0.37645], std=[0.22803, 0.22145, 0.216989])
])

evaltrans = v2.Compose([
    v2.Resize([112, 112], antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.43216, 0.394666, 0.37645], std=[0.22803, 0.22145, 0.216989])
])

# Basketball-51 names clips as <label>_<video number>_<timestamp>_x264.mp4.
# the paper defines the video number as the source game, e.g. 2p0_v108_000437_x264.mp4.
GAME_PATTERN = re.compile(r"_(v\d+)_")

def game_id_from_path(path):
    match = GAME_PATTERN.search(path.stem)
    if match is None:
        raise ValueError(f"Could not extract a game id from {path.name}")
    return match.group(1)

def discover_samples(datadir, samplesperclass):
    datadir = Path(datadir)
    classes = sorted([d.name for d in datadir.iterdir() if d.is_dir() and not d.name.startswith(".")])
    videopaths, labels, game_ids = [], [], []

    for label, name in enumerate(classes):
        classdir = datadir / name
        vids = sorted([f for f in classdir.iterdir() if f.is_file() and f.suffix.lower() == ".mp4"])
        for path in vids[:samplesperclass]:
            videopaths.append(path)
            labels.append(label)
            game_ids.append(game_id_from_path(path))

    return classes, np.asarray(videopaths, dtype=object), np.asarray(labels), np.asarray(game_ids)

def make_grouped_splits(labels, game_ids):
    # Outer 10-fold split reserves approximately 10% for final testing.
    # An inner 6-fold split takes 1/6 of the remaining 90% for validation,
    # producing approximately 75/15/10 overall while keeping games disjoint.
    indices = np.arange(len(labels))
    if len(np.unique(game_ids)) < 10:
        raise ValueError("grouped 75/15/10 splitting requires at least 10 games.")

    outer = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
    trainval_idx, test_idx = next(outer.split(indices, labels, groups=game_ids))

    inner = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=43)
    train_rel, val_rel = next(inner.split(
        trainval_idx, labels[trainval_idx], groups=game_ids[trainval_idx]
    ))
    train_idx = trainval_idx[train_rel]
    val_idx = trainval_idx[val_rel]

    split_indices = {"train": train_idx, "validation": val_idx, "test": test_idx}
    split_games = {name: set(game_ids[idx]) for name, idx in split_indices.items()}
    assert split_games["train"].isdisjoint(split_games["validation"])
    assert split_games["train"].isdisjoint(split_games["test"])
    assert split_games["validation"].isdisjoint(split_games["test"])
    return split_indices, split_games

class BasketballDataset(Dataset):
    def __init__(self, videopaths, labels, classes, training, frames=16):
        self.videopaths = list(videopaths)
        self.labels = list(labels)
        self.classes = classes
        self.goalframes = frames
        self.training = training
        print(f"loaded {len(self.videopaths)} video paths for {'training' if training else 'evaluation'}")

    def __len__(self):
        return len(self.videopaths)

    def __getitem__(self, idx):
        path = self.videopaths[idx]
        label = self.labels[idx]
        frames = self._load_video(path)

        vidtensor = torch.from_numpy(frames).permute(0, 3, 1, 2)
        vidtensor = traintrans(vidtensor) if self.training else evaltrans(vidtensor)
        vidtensor = vidtensor.permute(1, 0, 2, 3)
        return vidtensor, label

    def _load_video(self, path):
        cap = cv2.VideoCapture(str(path))
        framecount = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if framecount < self.goalframes:
            idxs = np.pad(np.arange(framecount), (0, self.goalframes - framecount), mode='edge')
        else:
            idxs = np.linspace(0, framecount - 1, self.goalframes, dtype=int)

        idxs = set(idxs)
        frames = []
        frameidx = 0

        while len(frames) < self.goalframes:
            ret, frame = cap.read()
            if ret:
                if frameidx in idxs:
                    frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            else:
                while len(frames) < self.goalframes:
                    fallback = frames[-1] if frames else np.zeros(
                        (int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)), int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), 3),
                        dtype=np.uint8
                    )
                    frames.append(fallback)
            frameidx += 1

        cap.release()
        return np.stack(frames)

In [ ]:
class_names, allpaths, alllabels, allgames = discover_samples(datasetpath, samplesperclass)
split_indices, split_games = make_grouped_splits(alllabels, allgames)

for split_name, idx in split_indices.items():
    class_counts = np.bincount(alllabels[idx], minlength=len(class_names))
    print(
        f"{split_name:10s}: {len(idx):5d} clips ({len(idx) / len(alllabels):.1%}), "
        f"{len(split_games[split_name]):2d} games, class counts {dict(zip(class_names, class_counts.tolist()))}"
    )

# save exact game assignments so every later experiment can reuse this split
split_manifest = {
    split_name: {
        "games": sorted(split_games[split_name]),
        "clips": [str(path.relative_to(Path(datasetpath))) for path in allpaths[idx]]
    }
    for split_name, idx in split_indices.items()
}
with open(os.path.join(savedir, "split_manifest.json"), "w") as f:
    json.dump(split_manifest, f, indent=2)

train_idx = split_indices["train"]
val_idx = split_indices["validation"]
test_idx = split_indices["test"]

traindata = BasketballDataset(allpaths[train_idx], alllabels[train_idx], class_names, True, frames)
valdata = BasketballDataset(allpaths[val_idx], alllabels[val_idx], class_names, False, frames)
testdata = BasketballDataset(allpaths[test_idx], alllabels[test_idx], class_names, False, frames)

loader_options = dict(
    batch_size=batchsize,
    num_workers=workers,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(workers > 0)
)
trainloader = DataLoader(traindata, shuffle=True, **loader_options)
valloader = DataLoader(valdata, shuffle=False, **loader_options)
testloader = DataLoader(testdata, shuffle=False, **loader_options)

classes = len(class_names)

In [ ]:
# # loading the R(2+1)D-18 model
weights = video_models.R2Plus1D_18_Weights.KINETICS400_V1
model = video_models.r2plus1d_18(weights=weights)

infeatures = model.fc.in_features
model.fc = nn.Linear(infeatures, classes)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.device_count() > 1:
    print(f"using {torch.cuda.device_count()} GPUs bless kaggle :praying_hands:")
    model = nn.DataParallel(model)

model = model.to(device)

print(f"model loaded on {device}")

In [ ]:
idxtoclass = {i: v for i,v in enumerate(traindata.classes)}

criterion = nn.CrossEntropyLoss(reduction="none")
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0002, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.7, patience=0)

currbestacc = 0
stale = 0

In [ ]:
def evaluate_model(loader, split_name):
    model.eval()
    class_loss = defaultdict(float)
    class_correct = defaultdict(float)
    class_total = defaultdict(float)
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for vids, labels in loader:
            vids = vids.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(vids)
            losses = criterion(outputs, labels)
            predicted = outputs.argmax(dim=1)

            total_loss += losses.sum().item()
            total_correct += (predicted == labels).sum().item()
            total_samples += labels.size(0)

            for label, loss, correct in zip(labels, losses, predicted == labels):
                k = label.item()
                class_loss[k] += loss.item()
                class_correct[k] += correct.item()
                class_total[k] += 1

    metrics = {
        "loss": total_loss / total_samples,
        "accuracy": total_correct / total_samples,
        "class_loss": [class_loss[k] / class_total[k] for k in range(classes)],
        "class_accuracy": [class_correct[k] / class_total[k] for k in range(classes)]
    }
    print(f"{split_name} loss: {metrics['loss']:.3f}")
    print(f"{split_name} accuracy: {metrics['accuracy']:.1%}")
    for k in range(classes):
        print(
            f"{idxtoclass[k]}: loss {metrics['class_loss'][k]:.3f}, "
            f"accuracy {metrics['class_accuracy'][k]:.1%}"
        )
    return metrics


history = []

for epoch in range(1, epochs + 1):
    print(f"\n==================== epoch {epoch}/{epochs} ====================")
    model.train()

    trainloss = defaultdict(float)
    traincorrect = defaultdict(float)
    traintotal = defaultdict(float)
    totaltrainloss = 0.0
    totaltraincorrect = 0
    totaltrainsamples = 0

    for batch_idx, (vids, labels) in enumerate(trainloader, start=1):
        vids = vids.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(vids)
        losses = criterion(outputs, labels)
        losses.mean().backward()
        optimizer.step()

        predicted = outputs.argmax(dim=1)
        totaltrainloss += losses.sum().item()
        totaltraincorrect += (predicted == labels).sum().item()
        totaltrainsamples += labels.size(0)

        for label, loss, correct in zip(labels, losses, predicted == labels):
            k = label.item()
            trainloss[k] += loss.item()
            traincorrect[k] += correct.item()
            traintotal[k] += 1

        if batch_idx % 50 == 0 or batch_idx == len(trainloader):
            print(
                f"batch {batch_idx}/{len(trainloader)}: "
                f"loss {totaltrainloss / totaltrainsamples:.3f}, "
                f"accuracy {totaltraincorrect / totaltrainsamples:.1%}"
            )

    valmetrics = evaluate_model(valloader, "validation")
    trainmetrics = [
        totaltrainloss / totaltrainsamples,
        totaltraincorrect / totaltrainsamples,
        [trainloss[k] / traintotal[k] for k in range(classes)],
        [traincorrect[k] / traintotal[k] for k in range(classes)]
    ]
    history.append(trainmetrics + [
        valmetrics["loss"], valmetrics["accuracy"],
        valmetrics["class_loss"], valmetrics["class_accuracy"]
    ])

    oldlr = optimizer.param_groups[0]["lr"]
    scheduler.step(valmetrics["accuracy"])
    newlr = optimizer.param_groups[0]["lr"]
    if newlr != oldlr:
        print(f"learning rate reduced from {oldlr} to {newlr}")

    if valmetrics["accuracy"] > currbestacc:
        stale = 0
        currbestacc = valmetrics["accuracy"]
        torch.save(model.state_dict(), bestpath)
        print(f"saved new best checkpoint with validation accuracy {currbestacc:.1%}")
    else:
        stale += 1
        if stale >= patience:
            print(f"no validation improvement after {patience} epochs; stopping")
            break

historypath = os.path.join(savedir, "history.json")
with open(historypath, "w") as f:
    json.dump(history, f)

In [ ]:
# final eval

try:
    best_state = torch.load(bestpath, map_location=device, weights_only=True)
except TypeError:
    best_state = torch.load(bestpath, map_location=device)
model.load_state_dict(best_state)

final_test_metrics = evaluate_model(testloader, "FINAL TEST")
with open(os.path.join(savedir, "final_test_metrics.json"), "w") as f:
    json.dump(final_test_metrics, f, indent=2)

In [ ]:
# visualizations

from matplotlib import pyplot as plt

# train and test loss over epochs

plt.plot([i for i in range(1,len(history)+1)], [history[i][0] for i in range(len(history))], label="training", color="blue")
plt.plot([i for i in range(1,len(history)+1)], [history[i][4] for i in range(len(history))], label="validation", color="red")

plt.xlabel("Epochs")
plt.ylabel("Average Loss")
plt.title("Average Loss per Epoch")

plt.legend()
plt.savefig("loss")
plt.show()


# train and test accuracy over epochs

plt.plot([i for i in range(1,len(history)+1)], [history[i][1] for i in range(len(history))], label="training", color="blue", ls="--")
plt.plot([i for i in range(1,len(history)+1)], [history[i][5] for i in range(len(history))], label="validation", color="red", ls="--")

plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Accuracy per Epoch")

plt.legend()
plt.savefig("acc")
plt.show()


# loss of each class over epochs

colors = ["red","orange","yellow","green","blue","cyan","purple","black"]

# training
for j in range(classes):
    plt.plot([i for i in range(1,len(history)+1)], [history[i][2][j] for i in range(len(history))], label=f"{traindata.classes[j]}", color=colors[j], ls="-")

plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Average Loss per Class per Epoch (training)")

plt.legend()
plt.savefig("classlosstrain")
plt.show()

# validation
for j in range(classes):
    plt.plot([i for i in range(1,len(history)+1)], [history[i][6][j] for i in range(len(history))], label=f"{testdata.classes[j]}", color=colors[j], ls="--")

plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Average Loss per Class per Epoch (validation)")

plt.legend()
plt.savefig("classlossval")
plt.show()


# accuracy of each class over epochs

# training
for j in range(classes):
    plt.plot([i for i in range(1,len(history)+1)], [history[i][3][j] for i in range(len(history))], label=f"{traindata.classes[j]}", color=colors[j])

plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Accuracy per Class per Epoch (training)")

plt.legend()
plt.savefig("classacctrain")
plt.show()

# validation
for j in range(classes):
    plt.plot([i for i in range(1,len(history)+1)], [history[i][7][j] for i in range(len(history))], label=f"{testdata.classes[j]}", color=colors[j])

plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Accuracy per Class per Epoch (validation)")

plt.legend()
plt.savefig("classaccval")
plt.show()